# Exercise 4: RF to analytic and baseband IQ

## Question

How do two ways of making complex RF data compare in accuracy, memory, and time on CPU and GPU—and when does parallel CPU conversion help?

We use two measured datasets with very different trace lengths, keep file loading outside the conversion timers, and pass both conversion outputs through Mach with the matching modulation-frequency setting. The notebook is prepared for a VS Code kernel connected to a Colab T4.


## Methods and interpretation

| Representation | CPU | GPU | Mach modulation frequency |
|---|---|---|---|
| Analytic RF | `scipy.signal.hilbert` | `cupyx.scipy.signal.hilbert` | `0 Hz` |
| Baseband IQ | PyMUST-style complex downmix + 5th-order Butterworth low-pass | CuPy/CuPyX equivalent | Dataset center frequency |

Hilbert creates an analytic signal while retaining its RF carrier. Downmixing shifts that carrier to zero frequency and filters away the unwanted sideband. For a fair comparison, the notebook demodulates the analytic signal by the known carrier before comparing complex samples, and it compares B-mode magnitudes after beamforming with the correct Mach setting for each representation.

The PyMUST rotating-disk recording is a short slow-time ensemble. PICMUS contains a much longer fast-time record for each of 75 independent transmit angles and one imaging frame. The PICMUS modulation-frequency field is zero for its real raw RF; its actual carrier is read from `channel_data.pulse.center_frequency`. That zero tells Mach to make analytic RF in its [UFF adapter](https://github.com/Forest-Neurotech/mach/blob/v0.2.1/src/mach/io/uff.py#L671-L725). The notebook bypasses that adapter preprocessing for the timed paths and records geometry setup separately from raw file loading.


## 1. Configure the run

Open this notebook in VS Code, connect its kernel to the Colab T4, and run the install cell. If pip asks for a kernel restart, restart it before continuing. The notebook uses a vectorized CPU baseline and a small CPU worker sweep over complete receive traces; the fast-time samples of a trace stay together.

The timing report separates cache/download and UFF loading, input upload, conversion compute, output download, and transfer-inclusive conversion-to-image time. CPU worker counts, data sizes, and sampled peak memory deltas are reported alongside CPU/GPU numerical differences. Memory deltas are sampled estimates; exact raw and output payload sizes are recorded separately. The last section can add a deterministic synthetic scale case if the two real data sizes do not show the crossover clearly.


In [ ]:
from pathlib import Path
import os

RUN_CPU_WORKER_SWEEP = True
RUN_BEAMFORM_IMAGE_CHECKS = True
RUN_SYNTHETIC_SCALE_CASE = False  # Enable only if a larger scale point is useful.
CPU_WORKERS = (1, 2, 4)
WARMUP_RUNS = 1
MEASURED_RUNS = 3
SYNTHETIC_SCALE_MULTIPLIER = 2

_candidates = [Path("/content/realtime-beamforming-lab"), Path.cwd(), *Path.cwd().resolve().parents]
ROOT = next((p.resolve() for p in _candidates if (p / "pyproject.toml").is_file()), Path.cwd().resolve())
RESULTS_DIR = ROOT / "results"
ARTIFACTS_DIR = ROOT / "artifacts"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Kernel working directory:", ROOT)
print("Results:", RESULTS_DIR)
print("Artifacts:", ARTIFACTS_DIR)
print("CPU workers:", CPU_WORKERS)


### Install the Colab dependencies

This installs the Colab CUDA extra used by the lab and removes conflicting CuPy CUDA wheels first. Run it on the connected T4, then restart the kernel if requested.


In [ ]:
%pip uninstall -y cupy-cuda12x cupy-cuda13x
%pip install --no-cache-dir "mach-beamform[examples,uff,vis]>=0.2.1,<0.3.0" "cupy-cuda13x>=13.0" "pymust>=0.1.9" "psutil>=5.9"


In [ ]:
import importlib.metadata as metadata
import platform
import subprocess
import sys

import cupy as cp
import numpy as np
import scipy

_gpu = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
print(_gpu.stdout if _gpu.returncode == 0 else _gpu.stderr)
if _gpu.returncode != 0 or cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError("No CUDA device is visible. Connect the notebook to a Colab T4 GPU runtime.")

_device = cp.cuda.runtime.getDeviceProperties(0)
_gpu_name = _device["name"].decode() if isinstance(_device["name"], bytes) else str(_device["name"])
print("Python:", platform.python_version())
print("GPU:", _gpu_name)
for _package in ("mach-beamform", "cupy-cuda13x", "PyMUST", "scipy", "pyuff-ustb"):
    try:
        print(f"{_package}: {metadata.version(_package)}")
    except metadata.PackageNotFoundError:
        print(f"{_package}: version metadata unavailable")


## 2. Load raw RF and acquisition metadata

The dataset loader and array layout conversion happen once. PICMUS carrier frequency comes from the UFF pulse metadata, while its zero modulation-frequency field is retained as the Mach setting for analytic RF. The baseband path uses the pulse center frequency.


In [ ]:
import hashlib
import time

from mach.io.must import download_pymust_doppler_data, extract_pymust_params
from mach.io.utils import cached_download
from pyuff_ustb import Uff


def scalar(value, name):
    array = np.asarray(value).squeeze()
    if array.size != 1:
        raise ValueError(f"Expected scalar metadata for {name}; got shape {array.shape}")
    return float(array)


def array_fingerprint(array):
    contiguous = np.ascontiguousarray(array)
    digest = hashlib.sha256()
    digest.update(contiguous.dtype.str.encode("ascii"))
    digest.update(np.asarray(contiguous.shape, dtype="<i8").tobytes())
    digest.update(contiguous.tobytes(order="C"))
    return digest.hexdigest()

# PyMUST PWI_disk.mat: loader verifies the upstream file hash.
_load_start = time.perf_counter_ns()
pymust_mat = download_pymust_doppler_data()
pymust_params = extract_pymust_params(pymust_mat)
pymust_rf_original = np.asarray(pymust_mat["RF"])
pymust_load_ms = (time.perf_counter_ns() - _load_start) / 1e6

pymust_rf = np.ascontiguousarray(pymust_rf_original, dtype=np.float32)
pymust_meta = {
    "fs_hz": scalar(pymust_params["fs"], "PyMUST fs"),
    "fc_hz": scalar(pymust_params["fc"], "PyMUST fc"),
    "t0_s": scalar(pymust_params.get("t0", 0.0), "PyMUST t0"),
    "bandwidth_percent": (
        scalar(pymust_params["bandwidth"], "PyMUST bandwidth")
        if pymust_params.get("bandwidth", None) is not None else None
    ),
    "_pymust_params": pymust_params,
}

# PICMUS experimental resolution/distortion UFF.
PICMUS_URL = "https://zenodo.org/records/20261898/files/PICMUS_experiment_resolution_distortion.uff"
PICMUS_FILENAME = "PICMUS_experiment_resolution_distortion.uff"
PICMUS_SIZE = 145_518_524
PICMUS_SHA256 = "c93af0781daeebf771e53a42a629d4f311407410166ef2f1d227e9d2a1b8c641"
_download_start = time.perf_counter_ns()
picmus_path = cached_download(
    PICMUS_URL, expected_size=PICMUS_SIZE, expected_hash=PICMUS_SHA256,
    digest=hashlib.sha256, filename=PICMUS_FILENAME,
)
picmus_download_cache_ms = (time.perf_counter_ns() - _download_start) / 1e6
_parse_start = time.perf_counter_ns()
uff = Uff(str(picmus_path))
picmus_channel_data = uff.read("/channel_data")
picmus_scan = uff.read("/scan")
picmus_rf_original = np.asarray(picmus_channel_data.data)
if picmus_rf_original.ndim == 3:
    picmus_rf_original = picmus_rf_original[..., None]
picmus_rf = np.ascontiguousarray(picmus_rf_original, dtype=np.float32)
picmus_parse_array_ms = (time.perf_counter_ns() - _parse_start) / 1e6

picmus_carrier_hz = float(picmus_channel_data.pulse.center_frequency)
picmus_modulation_hz = float(picmus_channel_data.modulation_frequency)
if not np.isfinite(picmus_carrier_hz) or picmus_carrier_hz <= 0:
    raise ValueError("PICMUS pulse.center_frequency is missing or invalid; verify UFF metadata before conversion.")
if picmus_modulation_hz != 0.0:
    raise ValueError(
        f"Expected real raw PICMUS RF with modulation_frequency=0 Hz; got {picmus_modulation_hz}. "
        "Inspect the dataset representation before continuing."
    )

picmus_meta = {
    "fs_hz": float(picmus_channel_data.sampling_frequency),
    "fc_hz": picmus_carrier_hz,
    "t0_s": float(picmus_channel_data.initial_time),
    "bandwidth_percent": None,  # Use PyMUST's default low-pass cutoff rule for this comparison.
}

datasets = {
    "pymust": {
        "label": "PyMUST rotating-disk phantom",
        "raw": pymust_rf,
        "rf_matrix": np.ascontiguousarray(pymust_rf.reshape(pymust_rf.shape[0], -1)),
        "meta": pymust_meta,
        "original": pymust_rf_original,
        "pymust_params": pymust_params,
        "load_ms": float(pymust_load_ms),
        "file_sha256": "c349dc1d677c561434fd0e4a74142c4a0a44b7e6ae0a42d446c078a528ef58c1",
        "rf_array_sha256": array_fingerprint(pymust_rf),
    },
    "picmus": {
        "label": "PICMUS experimental resolution/distortion phantom",
        "raw": picmus_rf,
        "rf_matrix": np.ascontiguousarray(picmus_rf.reshape(picmus_rf.shape[0], -1)),
        "meta": picmus_meta,
        "original": picmus_rf_original,
        "channel_data": picmus_channel_data,
        "scan": picmus_scan,
        "load_ms": float(picmus_download_cache_ms + picmus_parse_array_ms),
        "download_cache_ms": float(picmus_download_cache_ms),
        "uff_parse_array_ms": float(picmus_parse_array_ms),
        "file_sha256": PICMUS_SHA256,
        "rf_array_sha256": array_fingerprint(picmus_rf),
        "uff_modulation_frequency_hz": picmus_modulation_hz,
    },
}

for _name, _item in datasets.items():
    _rf = _item["raw"]
    print(
        f"{_name}: raw shape={_rf.shape}, traces={_item['rf_matrix'].shape[1]:,}, "
        f"dtype={_rf.dtype}, bytes={_rf.nbytes / 1e6:.1f} MB, "
        f"fs={_item['meta']['fs_hz']/1e6:.3f} MHz, fc={_item['meta']['fc_hz']/1e6:.3f} MHz, "
        f"load/cache={_item['load_ms']:.1f} ms"
    )
print("PICMUS pulse center frequency:", picmus_carrier_hz / 1e6, "MHz")
print("PICMUS modulation-frequency metadata:", picmus_modulation_hz, "Hz")


## 3. Implement matched conversion paths

For the PyMUST dataset, the vectorized single-worker CPU baseline calls `pymust.rf2iq`; its worker sweep applies the same operations to independent trace chunks. PICMUS uses the matching SciPy implementation. The baseband path follows [PyMUST's `rf2iq` implementation](https://github.com/creatis-ULTIM/PyMUST/blob/fe73cc3911a51b48b36095c2e0ad29c266532447/src/pymust/rf2iq.py): complex mixing at the carrier, a fifth-order Butterworth low-pass, zero-phase forward/backward filtering, and a factor of two for envelope amplitude. The GPU Hilbert implementation is [CuPy's signal API](https://docs.cupy.dev/en/stable/reference/generated/cupyx.scipy.signal.hilbert.html). Its cutoff follows PyMUST's optional-bandwidth rule when PyMUST metadata supplies a bandwidth; otherwise it uses the default `min(2 fc / fs, 0.5)` normalized cutoff. Both outputs are stored as complex64 for the CPU/GPU comparison.

CPU workers receive disjoint columns, so each complete fast-time trace remains intact.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import gc
import threading

from scipy import signal as scipy_signal


def filter_cutoff(meta):
    fs_hz, fc_hz = meta["fs_hz"], meta["fc_hz"]
    bandwidth = meta.get("bandwidth_percent")
    if bandwidth is None:
        wn = min(2.0 * fc_hz / fs_hz, 0.5)
    else:
        wn = (fc_hz * bandwidth / 100.0) / fs_hz
    if not (0.0 < wn <= 1.0):
        raise ValueError(f"Invalid normalized Butterworth cutoff {wn}")
    return float(wn)


def cpu_convert(method, rf_matrix, meta):
    if method == "hilbert":
        result = scipy_signal.hilbert(rf_matrix, axis=0)
    elif method == "baseband":
        fs_hz, fc_hz, t0_s = meta["fs_hz"], meta["fc_hz"], meta["t0_s"]
        t0 = np.asarray(t0_s, dtype=np.float64)
        if t0.size == 1:
            t = np.arange(rf_matrix.shape[0], dtype=np.float64) / fs_hz + float(t0.reshape(-1)[0])
        elif t0.size == rf_matrix.shape[0]:
            t = np.arange(rf_matrix.shape[0], dtype=np.float64) / fs_hz + t0.reshape(-1)
        else:
            raise ValueError("t0 must be scalar or one value per fast-time sample")
        mixer = np.exp(-2j * np.pi * fc_hz * t).reshape(-1, 1)
        mixed = rf_matrix * mixer
        b, a = scipy_signal.butter(5, filter_cutoff(meta))
        result = scipy_signal.filtfilt(b, a, mixed, axis=0) * 2.0
    else:
        raise ValueError(method)
    return np.ascontiguousarray(result, dtype=np.complex64)


def cpu_convert_workers(method, rf_matrix, meta, workers):
    workers = min(int(workers), rf_matrix.shape[1])
    if workers <= 1:
        if method == "baseband" and "_pymust_params" in meta:
            import pymust
            result = pymust.rf2iq(rf_matrix, meta["_pymust_params"])
            return np.ascontiguousarray(result, dtype=np.complex64)
        return cpu_convert(method, rf_matrix, meta)
    bounds = np.linspace(0, rf_matrix.shape[1], workers + 1, dtype=int)
    chunks = [(int(bounds[i]), int(bounds[i + 1])) for i in range(workers)]
    with ThreadPoolExecutor(max_workers=workers) as pool:
        parts = list(pool.map(lambda pair: cpu_convert(method, rf_matrix[:, pair[0]:pair[1]], meta), chunks))
    return np.ascontiguousarray(np.concatenate(parts, axis=1), dtype=np.complex64)


def gpu_convert(method, rf_device, meta):
    from cupyx.scipy import signal as cupy_signal
    if method == "hilbert":
        result = cupy_signal.hilbert(rf_device, axis=0)
    elif method == "baseband":
        fs_hz, fc_hz, t0_s = meta["fs_hz"], meta["fc_hz"], meta["t0_s"]
        t0 = np.asarray(t0_s, dtype=np.float64)
        if t0.size == 1:
            t = cp.arange(rf_device.shape[0], dtype=cp.float64) / fs_hz + float(t0.reshape(-1)[0])
        elif t0.size == rf_device.shape[0]:
            t = cp.arange(rf_device.shape[0], dtype=cp.float64) / fs_hz + cp.asarray(t0.reshape(-1))
        else:
            raise ValueError("t0 must be scalar or one value per fast-time sample")
        mixer = cp.exp(-2j * cp.pi * fc_hz * t).astype(cp.complex64).reshape(-1, 1)
        mixed = rf_device * mixer
        b, a = cupy_signal.butter(5, filter_cutoff(meta))
        result = cupy_signal.filtfilt(b, a, mixed, axis=0) * 2.0
    else:
        raise ValueError(method)
    return cp.ascontiguousarray(result, dtype=cp.complex64)


def percentile_summary(samples):
    values = np.asarray(samples, dtype=np.float64)
    return {"samples_ms": values.tolist(), "p50_ms": float(np.percentile(values, 50)), "p95_ms": float(np.percentile(values, 95))}


def time_cpu(method, rf_matrix, meta, workers):
    for _ in range(WARMUP_RUNS):
        cpu_convert_workers(method, rf_matrix, meta, workers)
    samples, latest = [], None
    for _ in range(MEASURED_RUNS):
        started = time.perf_counter_ns()
        latest = cpu_convert_workers(method, rf_matrix, meta, workers)
        samples.append((time.perf_counter_ns() - started) / 1e6)
    return percentile_summary(samples), latest


def time_gpu(method, rf_device, meta):
    # Capture first-call planning/JIT overhead separately from steady-state samples.
    started = time.perf_counter_ns()
    cold = gpu_convert(method, rf_device, meta)
    cp.cuda.Device().synchronize()
    cold_wall_ms = (time.perf_counter_ns() - started) / 1e6
    del cold
    for _ in range(WARMUP_RUNS):
        gpu_convert(method, rf_device, meta)
    cp.cuda.Device().synchronize()
    wall, device, latest = [], [], None
    for _ in range(MEASURED_RUNS):
        start_event, end_event = cp.cuda.Event(), cp.cuda.Event()
        started = time.perf_counter_ns()
        start_event.record()
        latest = gpu_convert(method, rf_device, meta)
        end_event.record()
        cp.cuda.Device().synchronize()
        wall.append((time.perf_counter_ns() - started) / 1e6)
        device.append(float(cp.cuda.get_elapsed_time(start_event, end_event)))
    return {"cold_first_call_wall_ms": float(cold_wall_ms), "wall": percentile_summary(wall), "device_event": percentile_summary(device)}, latest


def current_rss_mib():
    with open("/proc/self/statm", "r", encoding="ascii") as handle:
        resident_pages = int(handle.read().split()[1])
    return resident_pages * os.sysconf("SC_PAGE_SIZE") / (1024 ** 2)


def sampled_peak_memory(fn, device=False, interval_s=0.002):
    gc.collect()
    base_host = current_rss_mib()
    base_device = cp.get_default_memory_pool().used_bytes() if device else 0
    stop = threading.Event()
    peak_host, peak_device = [base_host], [base_device]
    def sample():
        while not stop.wait(interval_s):
            peak_host[0] = max(peak_host[0], current_rss_mib())
            if device:
                peak_device[0] = max(peak_device[0], cp.get_default_memory_pool().used_bytes())
    thread = threading.Thread(target=sample, daemon=True)
    thread.start()
    result = fn()
    if device:
        cp.cuda.Device().synchronize()
        peak_device[0] = max(peak_device[0], cp.get_default_memory_pool().used_bytes())
    peak_host[0] = max(peak_host[0], current_rss_mib())
    stop.set()
    thread.join()
    memory = {"sampled_host_rss_delta_mib": float(max(0.0, peak_host[0] - base_host))}
    if device:
        memory["sampled_cupy_pool_peak_delta_mib"] = float(max(0, peak_device[0] - base_device) / (1024 ** 2))
    return memory, result


def normalized_l2(reference, candidate):
    reference = np.asarray(reference)
    candidate = np.asarray(candidate)
    denominator = float(np.linalg.norm(reference.ravel().astype(np.complex128)))
    numerator = float(np.linalg.norm((reference - candidate).ravel().astype(np.complex128)))
    return numerator / denominator if denominator else 0.0

print("Conversion helpers ready; no conversion has been timed yet.")


## 4. Run conversion comparisons and beamformed image checks

For each dataset, the cell below records CPU worker-count timings, GPU event and wall timings, transfer-inclusive one-shot costs, sampled memory deltas, and numerical errors. The PyMUST dataset also checks the vectorized CPU baseband result against `pymust.rf2iq`. The image check uses the same GPU beamformer for all converted arrays, so its purpose is to compare representations and conversion accuracy rather than CPU versus GPU beamforming.


In [ ]:
from mach import wavefront
from mach.experimental import beamform as compound_beamform
from mach.io.must import linear_probe_positions, scan_grid
from mach.io.uff import compute_tx_wave_arrivals_s, extract_wave_directions
from mach.kernel import beamform as single_beamform


def prepare_gpu_input(rf_matrix):
    started = time.perf_counter_ns()
    device = cp.asarray(rf_matrix)
    cp.cuda.Device().synchronize()
    return device, (time.perf_counter_ns() - started) / 1e6


def pymust_beamform_setup(params, grid_size=251):
    nelements = int(params["Nelements"])
    pitch, sound_speed = float(params["pitch"]), float(params["c"])
    rx = linear_probe_positions(nelements, pitch)
    x = np.linspace(-12.5e-3, 12.5e-3, grid_size, endpoint=True, dtype=np.float32)
    y = np.array([0.0], dtype=np.float32)
    z = np.linspace(10e-3, 35e-3, grid_size, endpoint=True, dtype=np.float32)
    points = np.ascontiguousarray(scan_grid(x, y, z), dtype=np.float32)
    arrivals = wavefront.plane(
        origin_m=np.array([0.0, 0.0, 0.0], dtype=np.float32),
        points_m=points, direction=np.array([0.0, 0.0, 1.0], dtype=np.float32),
    ) / sound_speed
    return {
        "rx": cp.asarray(rx), "points": cp.asarray(points), "tx": cp.asarray(arrivals, dtype=cp.float32),
        "grid_shape": (len(x), len(z)), "f_number": float(params["fnumber"]),
        "rx_start_s": float(params["t0"]), "fs_hz": float(params["fs"]),
        "sound_speed_m_s": sound_speed,
    }


def picmus_beamform_setup(channel_data, scan):
    directions = extract_wave_directions(channel_data.sequence, np)
    points = np.ascontiguousarray(scan.xyz[0], dtype=np.float32)
    arrivals = compute_tx_wave_arrivals_s(
        directions, points, float(channel_data.sound_speed), xp=np
    )
    arrivals += np.asarray([wave.delay for wave in channel_data.sequence], dtype=np.float32)[:, None]
    rx = np.ascontiguousarray(channel_data.probe.xyz, dtype=np.float32)
    return {
        "rx": cp.asarray(rx), "points": cp.asarray(points), "tx": cp.asarray(arrivals, dtype=cp.float32),
        "grid_shape": (int(scan.x_axis.size), int(scan.z_axis.size)),
        "f_number": 1.7, "rx_start_s": float(channel_data.initial_time),
        "fs_hz": float(channel_data.sampling_frequency), "sound_speed_m_s": float(channel_data.sound_speed),
        "sequence_count": len(channel_data.sequence),
    }


def render_pymust(iq_matrix, dataset, modulation_hz, setup):
    # iq_matrix is samples × (elements * frames). Reshape to mach's elements × samples × frames.
    samples, elements, frames = dataset["raw"].shape
    iq = iq_matrix.reshape(samples, elements, frames).transpose(1, 0, 2)
    if isinstance(iq, cp.ndarray):
        iq_device, upload_ms = cp.ascontiguousarray(iq), 0.0
    else:
        upload_start = time.perf_counter_ns()
        iq_device = cp.ascontiguousarray(cp.asarray(iq))
        cp.cuda.Device().synchronize()
        upload_ms = (time.perf_counter_ns() - upload_start) / 1e6
    started = time.perf_counter_ns()
    beamformed = single_beamform(
        channel_data=iq_device, rx_coords_m=setup["rx"], scan_coords_m=setup["points"],
        tx_wave_arrivals_s=setup["tx"], f_number=setup["f_number"],
        rx_start_s=setup["rx_start_s"], sampling_freq_hz=setup["fs_hz"],
        sound_speed_m_s=setup["sound_speed_m_s"], modulation_freq_hz=modulation_hz,
        tukey_alpha=0.0,
    )
    cp.cuda.Device().synchronize()
    beamform_ms = (time.perf_counter_ns() - started) / 1e6
    envelope = cp.sqrt(cp.mean(cp.abs(beamformed) ** 2, axis=-1))
    image_d2h_start = time.perf_counter_ns()
    image = cp.asnumpy(envelope).reshape(setup["grid_shape"])
    cp.cuda.Device().synchronize()
    image_d2h_ms = (time.perf_counter_ns() - image_d2h_start) / 1e6
    return image, float(upload_ms), float(beamform_ms), float(image_d2h_ms)


def render_picmus(iq_matrix, dataset, modulation_hz, setup):
    # UFF order is samples × elements × transmit × frame; mach order is transmit × elements × samples × frame.
    samples, elements, transmits, frames = dataset["raw"].shape
    iq = iq_matrix.reshape(samples, elements, transmits, frames).transpose(2, 1, 0, 3)
    if isinstance(iq, cp.ndarray):
        iq_device, upload_ms = cp.ascontiguousarray(iq), 0.0
    else:
        upload_start = time.perf_counter_ns()
        iq_device = cp.ascontiguousarray(cp.asarray(iq))
        cp.cuda.Device().synchronize()
        upload_ms = (time.perf_counter_ns() - upload_start) / 1e6
    kwargs = {
        "channel_data": iq_device, "rx_coords_m": setup["rx"], "scan_coords_m": setup["points"],
        "tx_wave_arrivals_s": setup["tx"], "out": None, "f_number": setup["f_number"],
        "sampling_freq_hz": setup["fs_hz"], "sound_speed_m_s": setup["sound_speed_m_s"],
        "modulation_freq_hz": float(modulation_hz), "rx_start_s": setup["rx_start_s"],
    }
    started = time.perf_counter_ns()
    beamformed = compound_beamform(**kwargs)
    cp.cuda.Device().synchronize()
    beamform_ms = (time.perf_counter_ns() - started) / 1e6
    image_d2h_start = time.perf_counter_ns()
    image = cp.asnumpy(cp.abs(beamformed)).reshape(setup["grid_shape"])
    cp.cuda.Device().synchronize()
    image_d2h_ms = (time.perf_counter_ns() - image_d2h_start) / 1e6
    return image, float(upload_ms), float(beamform_ms), float(image_d2h_ms)


def image_metrics(reference, candidate):
    ref = np.asarray(reference, dtype=np.float64)
    got = np.asarray(candidate, dtype=np.float64)
    scale = max(float(np.max(ref)), float(np.max(got)), np.finfo(np.float64).tiny)
    diff = got - ref
    return {
        "normalized_l2_error": float(np.linalg.norm(diff) / max(np.linalg.norm(ref), np.finfo(np.float64).tiny)),
        "max_abs_error_over_shared_peak": float(np.max(np.abs(diff)) / scale),
        "correlation": float(np.corrcoef(ref.ravel(), got.ravel())[0, 1]),
    }




In [ ]:
report = {
    "exercise": 4, "gpu": _gpu_name, "python": platform.python_version(),
    "versions": {name: metadata.version(name) for name in ("mach-beamform", "cupy-cuda13x", "PyMUST", "scipy")},
    "cpu_logical_count": os.cpu_count(), "warmup_runs": WARMUP_RUNS, "measured_runs": MEASURED_RUNS,
    "cpu_workers_requested": list(CPU_WORKERS), "datasets": {},
    "method_notes": {
        "hilbert": "Analytic RF retains the carrier; Mach modulation frequency is 0 Hz.",
        "baseband": "Complex downmix, 5th-order Butterworth filtfilt, factor 2; Mach modulation frequency is the pulse center frequency.",
        "timing_scope": "CPU/GPU conversion starts after raw RF is loaded and made contiguous; transfer-inclusive GPU time adds raw H2D and converted output D2H.",
        "memory_scope": "Peak deltas are sampled every 2 ms and are approximate; exact raw/output payload bytes are recorded separately.",
        "beamforming_scope": "Image comparisons use the same GPU beamformer and report magnitude-image agreement. GPU-converted IQ stays resident through beamforming; CPU IQ upload is timed separately.",
    },
}

image_outputs = {}
for dataset_name, dataset in datasets.items():
    print(f"\n=== {dataset['label']} ===")
    rf = dataset["rf_matrix"]
    meta = dataset["meta"]
    dataset_result = {
        "label": dataset["label"], "raw_shape": list(dataset["raw"].shape),
        "fast_time_samples": int(rf.shape[0]), "independent_traces": int(rf.shape[1]),
        "raw_dtype": str(dataset["raw"].dtype), "raw_nbytes": int(dataset["raw"].nbytes),
        "rf_array_sha256": dataset["rf_array_sha256"], "source_file_sha256": dataset["file_sha256"],
        "sampling_frequency_hz": meta["fs_hz"], "carrier_frequency_hz": meta["fc_hz"],
        "initial_time_s": meta["t0_s"], "filter_cutoff_normalized": filter_cutoff(meta),
        "load_cache_ms": dataset["load_ms"], "methods": {}, "accuracy": {},
    }
    if dataset_name == "picmus":
        dataset_result["load_stages_ms"] = {"download_or_cache": dataset["download_cache_ms"], "uff_parse_and_array": dataset["uff_parse_array_ms"]}
        dataset_result["transmit_count"] = int(dataset["raw"].shape[2])
        dataset_result["frame_count"] = int(dataset["raw"].shape[3])
        dataset_result["uff_modulation_frequency_hz"] = dataset["uff_modulation_frequency_hz"]

    rf_device, h2d_ms = prepare_gpu_input(rf)
    setup = None
    render = None
    if RUN_BEAMFORM_IMAGE_CHECKS:
        setup_start = time.perf_counter_ns()
        if dataset_name == "pymust":
            setup = pymust_beamform_setup(dataset["pymust_params"])
            render = render_pymust
        else:
            setup = picmus_beamform_setup(dataset["channel_data"], dataset["scan"])
            render = render_picmus
        cp.cuda.Device().synchronize()
        dataset_result["beamforming_setup_ms_once"] = (time.perf_counter_ns() - setup_start) / 1e6

    cpu_outputs, images = {}, {}
    for method in ("hilbert", "baseband"):
        worker_results = {}
        baseline_output = None
        workers_to_run = CPU_WORKERS if RUN_CPU_WORKER_SWEEP else (1,)
        for worker_count in workers_to_run:
            timing, output = time_cpu(method, rf, meta, worker_count)
            worker_results[str(worker_count)] = timing
            if worker_count == 1:
                baseline_output = output
            del output
        if baseline_output is None:
            baseline_timing, baseline_output = time_cpu(method, rf, meta, 1)
            worker_results["1"] = baseline_timing

        memory_cpu, memory_output = sampled_peak_memory(
            lambda: cpu_convert(method, rf, meta), device=False
        )
        del memory_output
        gc.collect()
        cpu_outputs[method] = baseline_output
        worker_results["sampled_memory"] = memory_cpu

        cp.get_default_memory_pool().free_all_blocks()
        memory_gpu, memory_gpu_output = sampled_peak_memory(
            lambda: gpu_convert(method, rf_device, meta), device=True
        )
        del memory_gpu_output
        cp.get_default_memory_pool().free_all_blocks()
        timing_gpu, gpu_output = time_gpu(method, rf_device, meta)

        d2h_start = time.perf_counter_ns()
        gpu_output_host = cp.asnumpy(gpu_output)
        cp.cuda.Device().synchronize()
        d2h_ms = (time.perf_counter_ns() - d2h_start) / 1e6
        cpu_gpu_nrmse = normalized_l2(baseline_output, gpu_output_host)
        dataset_result["accuracy"][f"{method}_cpu_vs_gpu_complex_nrmse"] = cpu_gpu_nrmse

        method_result = {
            "cpu_workers": worker_results,
            "gpu": {
                "cold_first_call_wall_ms": timing_gpu["cold_first_call_wall_ms"],
                "wall": timing_gpu["wall"], "device_event": timing_gpu["device_event"],
                "raw_h2d_ms_once": float(h2d_ms), "converted_d2h_ms_once": float(d2h_ms),
                "transfer_inclusive_one_shot_ms_estimate": float(h2d_ms + timing_gpu["device_event"]["p50_ms"] + d2h_ms),
                "sampled_memory": memory_gpu,
            },
            "input_nbytes": int(rf.nbytes), "output_nbytes_complex64": int(baseline_output.nbytes),
        }

        if RUN_BEAMFORM_IMAGE_CHECKS:
            modulation_hz = 0.0 if method == "hilbert" else meta["fc_hz"]
            cpu_image, cpu_upload_ms, cpu_beamform_ms, cpu_image_d2h_ms = render(
                baseline_output, dataset, modulation_hz, setup
            )
            gpu_image, gpu_upload_ms, gpu_beamform_ms, gpu_image_d2h_ms = render(
                gpu_output, dataset, modulation_hz, setup
            )
            images[method] = {"cpu_conversion": cpu_image, "gpu_conversion": gpu_image}
            method_result["image_check"] = {
                "cpu_conversion_upload_ms": cpu_upload_ms, "cpu_conversion_beamform_ms": cpu_beamform_ms,
                "cpu_conversion_image_d2h_ms": cpu_image_d2h_ms,
                "gpu_conversion_upload_ms": gpu_upload_ms, "gpu_conversion_beamform_ms": gpu_beamform_ms,
                "gpu_conversion_image_d2h_ms": gpu_image_d2h_ms,
                "cpu_vs_gpu_conversion_image": image_metrics(cpu_image, gpu_image),
                "cpu_conversion_to_image_wall_ms_estimate": float(
                    worker_results["1"]["p50_ms"] + cpu_upload_ms + cpu_beamform_ms + cpu_image_d2h_ms
                ),
                "gpu_conversion_to_image_wall_ms_estimate": float(
                    h2d_ms + timing_gpu["wall"]["p50_ms"] + gpu_beamform_ms + gpu_image_d2h_ms
                ),
            }
        dataset_result["methods"][method] = method_result

        print(
            f"{method}: CPU p50={worker_results['1']['p50_ms']:.2f} ms; "
            f"GPU event p50={timing_gpu['device_event']['p50_ms']:.2f} ms; "
            f"H2D+compute+D2H≈{method_result['gpu']['transfer_inclusive_one_shot_ms_estimate']:.2f} ms; "
            f"CPU/GPU complex NRMSE={cpu_gpu_nrmse:.3g}"
        )
        del gpu_output, gpu_output_host
        cp.get_default_memory_pool().free_all_blocks()

    # PyMUST reference check is separate from the timed vectorized CPU method.
    if dataset_name == "pymust":
        import pymust
        pymust_reference = np.asarray(
            pymust.rf2iq(dataset["original"], dataset["pymust_params"]), dtype=np.complex64
        ).reshape(rf.shape)
        dataset_result["accuracy"]["cpu_baseband_vs_pymust_rf2iq_nrmse"] = normalized_l2(
            pymust_reference, cpu_outputs["baseband"]
        )
        del pymust_reference

    # Demodulate analytic RF by the known carrier before comparing complex samples.
    t = np.arange(rf.shape[0], dtype=np.float64) / meta["fs_hz"] + meta["t0_s"]
    demod = np.exp(-2j * np.pi * meta["fc_hz"] * t).astype(np.complex64).reshape(-1, 1)
    analytic_aligned = cpu_outputs["hilbert"] * demod
    dataset_result["accuracy"]["cpu_hilbert_demodulated_vs_cpu_baseband_nrmse"] = normalized_l2(
        cpu_outputs["baseband"], analytic_aligned
    )
    if RUN_BEAMFORM_IMAGE_CHECKS:
        dataset_result["accuracy"]["cpu_hilbert_vs_cpu_baseband_bmode"] = image_metrics(
            images["hilbert"]["cpu_conversion"], images["baseband"]["cpu_conversion"]
        )
        image_outputs[dataset_name] = images

    report["datasets"][dataset_name] = dataset_result
    del cpu_outputs, rf_device
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()

print("\nConversion comparison complete.")


## 5. Optional deterministic scale case

The measured PICMUS workload already has about two orders of magnitude more raw samples than PyMUST. Leave this disabled unless you want to extend the size curve. When enabled, it creates a reproducible RF signal with a known low-frequency envelope and carrier, at twice the PICMUS trace count by default. The synthetic timing is reported separately from measured data.


In [ ]:
if RUN_SYNTHETIC_SCALE_CASE:
    picmus = datasets["picmus"]
    n_samples = picmus["rf_matrix"].shape[0]
    n_traces = picmus["rf_matrix"].shape[1] * SYNTHETIC_SCALE_MULTIPLIER
    fs_hz = picmus["meta"]["fs_hz"]
    fc_hz = picmus["meta"]["fc_hz"]
    t = np.arange(n_samples, dtype=np.float64) / fs_hz
    trace_phase = np.linspace(0.0, 2 * np.pi, n_traces, endpoint=False, dtype=np.float32)
    envelope = 0.7 + 0.3 * np.cos(2 * np.pi * 150_000 * t[:, None] + trace_phase[None, :])
    synthetic_rf = np.ascontiguousarray(
        envelope * np.cos(2 * np.pi * fc_hz * t[:, None] + trace_phase[None, :]),
        dtype=np.float32,
    )
    synthetic_meta = {"fs_hz": fs_hz, "fc_hz": fc_hz, "t0_s": 0.0, "bandwidth_percent": None}
    synthetic_device, synthetic_h2d_ms = prepare_gpu_input(synthetic_rf)
    synthetic_results = {"shape": list(synthetic_rf.shape), "raw_nbytes": int(synthetic_rf.nbytes), "h2d_ms": synthetic_h2d_ms, "methods": {}}
    for method in ("hilbert", "baseband"):
        cpu_timing, cpu_value = time_cpu(method, synthetic_rf, synthetic_meta, 1)
        gpu_timing, gpu_value = time_gpu(method, synthetic_device, synthetic_meta)
        gpu_value_host = cp.asnumpy(gpu_value)
        ground_truth_iq = envelope * np.exp(1j * trace_phase[None, :])
        central = slice(64, -64) if n_samples > 128 else slice(None)
        if method == "baseband":
            ground_truth_candidate = cpu_value
        else:
            demod = np.exp(-2j * np.pi * fc_hz * t).astype(np.complex64).reshape(-1, 1)
            ground_truth_candidate = cpu_value * demod
        synthetic_results["methods"][method] = {
            "cpu_p50_ms": cpu_timing["p50_ms"], "gpu_event_p50_ms": gpu_timing["device_event"]["p50_ms"],
            "cpu_gpu_nrmse": normalized_l2(cpu_value, gpu_value_host),
            "cpu_vs_known_baseband_nrmse_center": normalized_l2(ground_truth_iq[central], ground_truth_candidate[central]),
            "output_nbytes": int(cpu_value.nbytes),
        }
        del cpu_value, gpu_value, gpu_value_host
    report["synthetic_scale_case"] = synthetic_results
    del synthetic_rf, synthetic_device
    cp.get_default_memory_pool().free_all_blocks()
    print(synthetic_results)
else:
    report["synthetic_scale_case"] = {"run": False, "reason": "Measured PICMUS is the current large workload; enable the config flag if a larger size point is needed."}
    print("Synthetic scale case skipped; PICMUS is the current large measured workload.")


## 6. Save the report and review the comparisons

The JSON report includes dataset fingerprints, software/runtime versions, timing samples, worker counts, transfer costs, sampled memory deltas, and accuracy metrics. Large converted RF arrays are not saved; the report remains small and the input data stays in its upstream cache.


In [ ]:
import json

report_path = RESULTS_DIR / "exercise4-rf-to-iq.json"
report_path.write_text(json.dumps(report, indent=2, default=lambda value: value.item() if isinstance(value, np.generic) else str(value)) + "\n")
print("Saved:", report_path)

try:
    _git_root = next(parent for parent in (ROOT, *ROOT.parents) if (parent / ".git").exists())
    _git_rev = subprocess.run(["git", "rev-parse", "HEAD"], cwd=_git_root, text=True, capture_output=True, check=True).stdout.strip()
    report["git_revision"] = _git_rev
    report_path.write_text(json.dumps(report, indent=2, default=lambda value: value.item() if isinstance(value, np.generic) else str(value)) + "\n")
    print("Git revision:", _git_rev)
except (StopIteration, subprocess.CalledProcessError):
    print("Git revision unavailable from the kernel working directory.")


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
import pandas as pd

rows = []
for dataset_name, dataset_result in report["datasets"].items():
    for method, method_result in dataset_result["methods"].items():
        worker_keys = [key for key in method_result["cpu_workers"] if key.isdigit()]
        best_worker = min(worker_keys, key=lambda key: method_result["cpu_workers"][key]["p50_ms"])
        rows.append({
            "Dataset": dataset_name, "Method": method,
            "CPU 1 worker p50 (ms)": method_result["cpu_workers"]["1"]["p50_ms"],
            "CPU best p50 (ms)": method_result["cpu_workers"][best_worker]["p50_ms"],
            "Best workers": int(best_worker),
            "GPU event p50 (ms)": method_result["gpu"]["device_event"]["p50_ms"],
            "GPU transfer-inclusive (ms)": method_result["gpu"]["transfer_inclusive_one_shot_ms_estimate"],
            "CPU/GPU signal NRMSE": dataset_result["accuracy"][f"{method}_cpu_vs_gpu_complex_nrmse"],
            "Host RSS delta (MiB)": method_result["cpu_workers"]["sampled_memory"]["sampled_host_rss_delta_mib"],
            "GPU pool delta (MiB)": method_result["gpu"]["sampled_memory"]["sampled_cupy_pool_peak_delta_mib"],
        })
summary = pd.DataFrame(rows)
display(summary.round(4))

fig, axes = plt.subplots(1, len(report["datasets"]), figsize=(14, 5), squeeze=False, constrained_layout=True)
for ax, (dataset_name, dataset_result) in zip(axes[0], report["datasets"].items()):
    methods = ("hilbert", "baseband")
    positions, labels, values, colors = [], [], [], []
    for m_index, method in enumerate(methods):
        method_result = dataset_result["methods"][method]
        for worker in CPU_WORKERS if RUN_CPU_WORKER_SWEEP else (1,):
            key = str(worker)
            if key in method_result["cpu_workers"]:
                positions.append(m_index * 6 + worker - 1)
                labels.append(f"{method}\nCPU {worker}w")
                values.append(method_result["cpu_workers"][key]["p50_ms"])
                colors.append("#9aa7b2")
        positions.append(m_index * 6 + 4)
        labels.append(f"{method}\nGPU")
        values.append(method_result["gpu"]["device_event"]["p50_ms"])
        colors.append("#2878b5")
    bars = ax.bar(positions, values, color=colors, width=0.78)
    ax.set_xticks(positions, labels, rotation=35, ha="right")
    ax.set_ylabel("Conversion p50 (ms)")
    ax.set_title(dataset_result["label"])
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
fig.suptitle("RF conversion: CPU worker sweep and GPU compute time")
timing_plot_path = RESULTS_DIR / "exercise4-conversion-timing.png"
fig.savefig(timing_plot_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", timing_plot_path)


In [ ]:
if RUN_BEAMFORM_IMAGE_CHECKS:
    fig, axes = plt.subplots(len(image_outputs), 3, figsize=(14, 5 * len(image_outputs)), squeeze=False, constrained_layout=True)
    for row, (dataset_name, outputs) in enumerate(image_outputs.items()):
        hilbert_image = outputs["hilbert"]["cpu_conversion"]
        baseband_image = outputs["baseband"]["cpu_conversion"]
        peak = max(float(np.max(hilbert_image)), float(np.max(baseband_image)), np.finfo(np.float64).tiny)
        for col, (image, title) in enumerate(((hilbert_image, "Hilbert analytic RF"), (baseband_image, "Downmix + low-pass IQ"))):
            view = axes[row, col].imshow(20 * np.log10(np.maximum(image / peak, 1e-4)).T, origin="lower", cmap="magma", vmin=-40, vmax=0, aspect="auto")
            axes[row, col].set_title(f"{dataset_name}: {title}")
            axes[row, col].set_xlabel("Lateral index")
            axes[row, col].set_ylabel("Depth index")
        difference = np.abs(baseband_image - hilbert_image) / peak
        view = axes[row, 2].imshow(difference.T, origin="lower", cmap="viridis", aspect="auto")
        axes[row, 2].set_title(f"{dataset_name}: magnitude difference / shared peak")
        axes[row, 2].set_xlabel("Lateral index")
        axes[row, 2].set_ylabel("Depth index")
    image_plot_path = RESULTS_DIR / "exercise4-beamformed-image-comparison.png"
    fig.savefig(image_plot_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", image_plot_path)
    for dataset_name, dataset_result in report["datasets"].items():
        print(dataset_name, "CPU Hilbert vs baseband B-mode:", dataset_result["accuracy"]["cpu_hilbert_vs_cpu_baseband_bmode"])


## Interpretation

1. Does the GPU reduce conversion time after excluding transfers? Does that conclusion change when raw input upload and converted output download are included?
2. Which CPU worker count helps on the short PyMUST traces and the longer PICMUS traces? Does a worker count above the runtime's logical CPU count only add overhead?
3. Is each GPU result close to its CPU counterpart? Does the PyMUST CPU implementation agree with `pymust.rf2iq`?
4. Do the analytic-RF and baseband paths produce similar B-mode magnitude images after each uses the appropriate Mach modulation-frequency setting?
5. How do raw/output payload bytes and sampled host/GPU memory change with workload size? Enable the synthetic scale case only if the measured datasets do not show a clear size trend.

The Hilbert and downmix/filter outputs are different signal representations. Interpret their direct complex-signal error only after carrier demodulation; use the correctly configured beamformed magnitude images to judge whether the choice changes the final image.
